# Kaggriculture — Rule-Based Starter Agent (v3)

A diversified, trickle-selling, land-hungry rule-based bot for the [Kaggriculture](https://www.kaggle.com/competitions/kaggriculture) competition.

This builds on two earlier iterations, each fixing a real bug or blind spot found by actually running the environment locally rather than guessing:

- **v1** guessed at the observation/action schema (wrong keys, wrong action shapes) — a scaffold only.
- **v2** used the real schema and a single "best crop" heuristic. Benchmarked at **~$4.6K** average final bank vs. the built-in `starter` agent across 8 seeds.
- **v3** (this notebook) folds in three findings from the community notebook [*Kaggriculture, Visualized: What Every Crop Pays*](https://www.kaggle.com/code/georgymamarin/kaggriculture-visualized-what-every-crop-pays) by Georgy Mamarin:
  1. **Diversify across crops** — a single crop crashes its own market as you sell into it (melon's oversupply curve is quadratic; it hits the $1 floor after ~158 net units sold). Spreading planting across the top few crops by live profit/tile-day avoids self-inflicted price crashes.
  2. **Trickle-sell** — orders resolve unit by unit and the price slides while you sell. A full-shed dump keeps under a tenth of the price the first unit promised. This agent caps how much of one product it sells per turn.
  3. **Buy land aggressively, against real prices** — even the $4,000 third quadrant pays back in under a week at a modest $25/tile-day. This agent checks against the actual `LAND_PRICES` tier costs rather than a single guessed threshold.

Result: **~$17.8K** average final bank vs. `starter` across the same 8 seeds — roughly a 4x improvement over v2, with the gain holding consistently across seeds (not a lucky single run).


## Setup

In [1]:
import subprocess, sys
# Uncomment if kaggle-environments isn't already installed in this kernel:
# subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "kaggle-environments"], check=True)

from kaggle_environments import make
from kaggle_environments.envs.kaggriculture.kaggriculture import CROPS, LAND_ORDER, LAND_PRICES
import statistics

print("Crops:", list(CROPS))
print("Land order:", LAND_ORDER, "at", LAND_PRICES)

Crops: ['WHEAT', 'CARROT', 'TOMATO', 'STRAWBERRY', 'MELON']
Land order: ['NE', 'SW', 'SE'] at [1000, 2000, 4000]


## Why the v2 -> v3 jump: the harvest-maturity bug (context)

Worth keeping visible: v1 -> v2 fixed a bug where non-ongoing crops (wheat, carrot, melon) start with `yield_units: 1` the instant they're planted, but the engine still rejects `HARVEST` until `first_yield_day` has passed. Checking `yield_units > 0` alone got the agent stuck repeatedly attempting an invalid harvest on an immature plant instead of watering it — final bank crashed to ~$100. `_is_mature()` below guards against that by also checking `day - planted_day >= first_yield_day`.

In [2]:
# --------------------------------------------------------------------------
# Tunable constants
# --------------------------------------------------------------------------
CASH_RESERVE = 100            # never let planned spend eat into this buffer
TARGET_CREW_SIZE = 6          # hands to maintain daily; Fib hire cost keeps this cheap
SEED_BUY_BATCH = 3            # max seeds to queue per turn, per crop in the portfolio
PORTFOLIO_SIZE = 3            # spread planting across this many top crops
SELL_BATCH_CAP = 20           # max units of one product sold in a single turn (trickle, not dump)
PRICE_HISTORY_WINDOW = 15     # turns of price memory per product for the sell-timing rule
MIN_HISTORY_FOR_THRESHOLD = 4 # sell opportunistically until we have this many samples

_price_history = {}

def _update_price_history(product, price):
    hist = _price_history.setdefault(product, [])
    hist.append(price)
    if len(hist) > PRICE_HISTORY_WINDOW:
        hist.pop(0)

def _should_sell(product, price):
    hist = _price_history.get(product, [])
    if len(hist) < MIN_HISTORY_FOR_THRESHOLD:
        return True  # not enough data yet -- take the cash rather than hoard
    return price >= statistics.mean(hist)

## Crop economics — live profit/tile-day, not a static table

Same occupied-days accounting as the source notebook's `crop_econ()`, but evaluated against the *current* market price each turn instead of the base price.

In [3]:
def _one_shot_units(crop):
    c = CROPS[crop]
    window = range((c["max_yield_day"] + 1) // 2, c["max_yield_day"] + 1)
    return min(c["max_yield"], 1 + len(list(window)))

def _one_shot_days(crop):
    c = CROPS[crop]
    cap_day = (c["max_yield_day"] + 1) // 2 + c["max_yield"] - 2
    return max(c["first_yield_day"], min(cap_day, c["max_yield_day"]))

def _ongoing_days(crop):
    c = CROPS[crop]
    return [c["first_yield_day"] + k * c["interval"] for k in range(c["max_yield"])]

def _crop_profit_per_tile_day(crop, price):
    c = CROPS[crop]
    if c["ongoing"]:
        days = _ongoing_days(crop)
        units, occupied = c["max_yield"], days[-1]
    else:
        units, occupied = _one_shot_units(crop), _one_shot_days(crop)
    revenue = units * price
    profit = revenue - c["seed"]
    return profit / max(1, occupied)

def _crop_portfolio(money, prices):
    """Top PORTFOLIO_SIZE affordable crops by live profit/tile-day. Planting
    gets round-robined across this list -- the actual diversification."""
    scored = []
    for crop, info in CROPS.items():
        if info["seed"] > money - CASH_RESERVE:
            continue
        price = prices.get(crop, 0)
        scored.append((_crop_profit_per_tile_day(crop, price), crop))
    if not scored:
        return []
    scored.sort(reverse=True)
    return [crop for _, crop in scored[:PORTFOLIO_SIZE]]

## Movement and per-unit decisions

In [4]:
def _step_toward(fx, fy, tx, ty):
    if fx > tx: return "WEST"
    if fx < tx: return "EAST"
    if fy > ty: return "NORTH"
    if fy < ty: return "SOUTH"
    return None

def _is_mature(tile, day):
    """Non-ongoing crops start with yield_units=1 at planting, but HARVEST
    is rejected until first_yield_day passes -- see the note above."""
    crop_info = CROPS.get(tile.get("crop"))
    if crop_info is None:
        return False
    return day - tile.get("planted_day", day) >= crop_info["first_yield_day"]

def _scan_targets(farm, board_size, plantable_crops, day):
    """Every actionable tile, tagged by purpose. Shared across all units --
    each independently picks its nearest target from the same snapshot, so
    units may converge on one tile in a turn (a mild inefficiency: a second
    HARVEST/PLANT there is a silent no-op, not a crash)."""
    harvestable, needs_water, plantable = [], [], []
    for y in range(board_size):
        for x in range(board_size):
            tile = farm["tiles"][y][x]
            if tile is None:
                if plantable_crops:
                    plantable.append((x, y))
                continue
            if not isinstance(tile, dict):
                continue  # "LOCKED"
            if tile.get("kind") != "PLANT":
                continue  # ignore animal structures for this crop-only pass
            if tile.get("yield_units", 0) > 0 and _is_mature(tile, day):
                harvestable.append((x, y))
            elif not tile.get("watered_today", False):
                needs_water.append((x, y))
    return harvestable, needs_water, plantable

def _unit_action(pos, farm, board_size, day, assigned_crop, have_seed, plantable_crops):
    """Decide the single next action for one unit (farmer or a hand)."""
    fx, fy = pos
    tile = farm["tiles"][fy][fx]

    if (isinstance(tile, dict) and tile.get("kind") == "PLANT"
            and tile.get("yield_units", 0) > 0 and _is_mature(tile, day)):
        return ["HARVEST"]

    if isinstance(tile, dict) and tile.get("kind") == "PLANT" and not tile.get("watered_today", False):
        return ["WATER"]

    if tile is None and have_seed and assigned_crop:
        return ["PLANT", assigned_crop]

    harvestable, needs_water, plantable = _scan_targets(farm, board_size, plantable_crops, day)
    for group in (harvestable, needs_water, plantable):
        if not group:
            continue
        tx, ty = min(group, key=lambda t: abs(t[0] - fx) + abs(t[1] - fy))
        step = _step_toward(fx, fy, tx, ty)
        if step:
            return [step]
    return ["PASS"]

## The agent

In [5]:
def agent(obs, configuration=None):
    farms = obs.get("farms", [])
    player = obs.get("player", 0)
    private = obs.get("private", {}) or {}
    if not farms or player >= len(farms):
        return {"farmer": ["PASS"], "hands": [], "market": []}

    farm = farms[player]
    board_size = len(farm["tiles"])
    day = obs.get("day", 0)
    money = farm["money"]
    seeds = private.get("seeds", {})
    shed = private.get("shed", {})
    market_prices = (obs.get("market", {}) or {}).get("prices", {})

    for product, price in market_prices.items():
        _update_price_history(product, price)

    portfolio = _crop_portfolio(money, market_prices)
    market_orders = []

    # Trickle-sell instead of dumping the whole shed.
    for product, qty in shed.items():
        if qty > 0 and product in market_prices and _should_sell(product, market_prices[product]):
            market_orders.append(["SELL", product, min(qty, SELL_BATCH_CAP)])

    # Keep seed stock topped up across the whole portfolio.
    for crop in portfolio:
        if seeds.get(crop, 0) < SEED_BUY_BATCH and money - CASH_RESERVE >= CROPS[crop]["seed"]:
            market_orders.append(["BUY_SEED", crop, 1])

    # Re-hire up to target crew size every day (cost resets to cheap Fibonacci daily).
    if len(farm["hands"]) < TARGET_CREW_SIZE and money - CASH_RESERVE >= 1:
        market_orders.append(["HIRE"])

    # Buy the next real land tier as soon as affordable with reserve to spare.
    n_extra_unlocked = len(farm["unlocked_quadrants"]) - 1
    if n_extra_unlocked < len(LAND_ORDER):
        next_land_price = LAND_PRICES[n_extra_unlocked]
        if money - CASH_RESERVE >= next_land_price:
            market_orders.append(["BUY_LAND"])

    def crop_for_unit(idx):
        if not portfolio:
            return None
        return portfolio[idx % len(portfolio)]

    plantable_crops = bool(portfolio)

    farmer_crop = crop_for_unit(0)
    farmer_have_seed = bool(farmer_crop) and seeds.get(farmer_crop, 0) > 0
    farmer_action = _unit_action(farm["farmer"], farm, board_size, day, farmer_crop, farmer_have_seed, plantable_crops)

    hands_actions = []
    for i, hand_pos in enumerate(farm["hands"], start=1):
        hand_crop = crop_for_unit(i)
        hand_have_seed = bool(hand_crop) and seeds.get(hand_crop, 0) > 0
        hands_actions.append(_unit_action(hand_pos, farm, board_size, day, hand_crop, hand_have_seed, plantable_crops))

    return {"farmer": farmer_action, "hands": hands_actions, "market": market_orders}

## Sanity check: one game

In [6]:
env = make("kaggriculture", debug=True)
env.run([agent, "starter"])

final = env.steps[-1]
for i, s in enumerate(final):
    print(f"Player {i}: reward={s.reward}, status={s.status}")

Player 0: reward=18655.0, status=DONE
Player 1: reward=3495.0, status=DONE


## Benchmark across seeds

A single game is noisy — the town unlocks a random shop every 3 days, so two episodes hand the bot different demand entirely. Run several seeds and compare, as the source notebook recommends, rather than trusting one number.

In [7]:
for opponent in ["random", "starter", "pass"]:
    results = []
    for seed in range(8):
        env = make("kaggriculture", configuration={"seed": seed}, debug=False)
        env.run([agent, opponent])
        results.append(env.steps[-1][0]["reward"])
    avg = sum(results) / len(results)
    print(f"vs {opponent:8s}  seeds 0-7: {[round(r) for r in results]}  avg=${avg:,.0f}")

vs random    seeds 0-7: [17252, 18316, 18385, 17473, 18245, 16814, 17258, 18309]  avg=$17,756


vs starter   seeds 0-7: [17553, 18413, 18477, 17908, 18383, 16207, 17599, 18134]  avg=$17,834


vs pass      seeds 0-7: [17285, 18358, 18011, 16963, 18575, 16757, 17461, 19301]  avg=$17,839


## Next steps, in the source notebook's rough order of payoff

1. **Animals** — a goose/cow/sheep is a compounding asset that also prints free fertilizer daily (the only way wheat reaches its 6-unit cap). Needs a stateful per-unit routine: `PICKUP` wheat from the shed -> walk to the animal -> `FEED` -> `CARE` -> `HARVEST` on production days -> walk home -> `DROP`.
2. **Fertilizer** on crops, not just animals — `FERTILIZE` doubles the watering-window yield bonus for 3 days.
3. **Smarter sell-price threshold** — right now it's a simple rolling average; the actual `market_price()` curve (imported above) could be inverted to estimate "how many units can I sell before the price drops below my threshold," making the trickle cap dynamic per product instead of one flat `SELL_BATCH_CAP`.
4. **Task allocation instead of round-robin** — assigning units to crops by index is simple but doesn't account for tile geography; a proper scheduler (like the community's "Diversified Scheduler Baseline") assigns the *nearest* unit to the *nearest* task instead.

## Export as `main.py` for submission

In [8]:
%%writefile main.py
import statistics
from kaggle_environments.envs.kaggriculture.kaggriculture import CROPS, LAND_ORDER, LAND_PRICES

CASH_RESERVE = 100
TARGET_CREW_SIZE = 6
SEED_BUY_BATCH = 3
PORTFOLIO_SIZE = 3
SELL_BATCH_CAP = 20
PRICE_HISTORY_WINDOW = 15
MIN_HISTORY_FOR_THRESHOLD = 4

_price_history = {}

def _update_price_history(product, price):
    hist = _price_history.setdefault(product, [])
    hist.append(price)
    if len(hist) > PRICE_HISTORY_WINDOW:
        hist.pop(0)

def _should_sell(product, price):
    hist = _price_history.get(product, [])
    if len(hist) < MIN_HISTORY_FOR_THRESHOLD:
        return True
    return price >= statistics.mean(hist)

def _one_shot_units(crop):
    c = CROPS[crop]
    window = range((c["max_yield_day"] + 1) // 2, c["max_yield_day"] + 1)
    return min(c["max_yield"], 1 + len(list(window)))

def _one_shot_days(crop):
    c = CROPS[crop]
    cap_day = (c["max_yield_day"] + 1) // 2 + c["max_yield"] - 2
    return max(c["first_yield_day"], min(cap_day, c["max_yield_day"]))

def _ongoing_days(crop):
    c = CROPS[crop]
    return [c["first_yield_day"] + k * c["interval"] for k in range(c["max_yield"])]

def _crop_profit_per_tile_day(crop, price):
    c = CROPS[crop]
    if c["ongoing"]:
        days = _ongoing_days(crop)
        units, occupied = c["max_yield"], days[-1]
    else:
        units, occupied = _one_shot_units(crop), _one_shot_days(crop)
    revenue = units * price
    profit = revenue - c["seed"]
    return profit / max(1, occupied)

def _crop_portfolio(money, prices):
    scored = []
    for crop, info in CROPS.items():
        if info["seed"] > money - CASH_RESERVE:
            continue
        price = prices.get(crop, 0)
        scored.append((_crop_profit_per_tile_day(crop, price), crop))
    if not scored:
        return []
    scored.sort(reverse=True)
    return [crop for _, crop in scored[:PORTFOLIO_SIZE]]

def _step_toward(fx, fy, tx, ty):
    if fx > tx: return "WEST"
    if fx < tx: return "EAST"
    if fy > ty: return "NORTH"
    if fy < ty: return "SOUTH"
    return None

def _is_mature(tile, day):
    crop_info = CROPS.get(tile.get("crop"))
    if crop_info is None:
        return False
    return day - tile.get("planted_day", day) >= crop_info["first_yield_day"]

def _scan_targets(farm, board_size, plantable_crops, day):
    harvestable, needs_water, plantable = [], [], []
    for y in range(board_size):
        for x in range(board_size):
            tile = farm["tiles"][y][x]
            if tile is None:
                if plantable_crops:
                    plantable.append((x, y))
                continue
            if not isinstance(tile, dict):
                continue
            if tile.get("kind") != "PLANT":
                continue
            if tile.get("yield_units", 0) > 0 and _is_mature(tile, day):
                harvestable.append((x, y))
            elif not tile.get("watered_today", False):
                needs_water.append((x, y))
    return harvestable, needs_water, plantable

def _unit_action(pos, farm, board_size, day, assigned_crop, have_seed, plantable_crops):
    fx, fy = pos
    tile = farm["tiles"][fy][fx]
    if (isinstance(tile, dict) and tile.get("kind") == "PLANT"
            and tile.get("yield_units", 0) > 0 and _is_mature(tile, day)):
        return ["HARVEST"]
    if isinstance(tile, dict) and tile.get("kind") == "PLANT" and not tile.get("watered_today", False):
        return ["WATER"]
    if tile is None and have_seed and assigned_crop:
        return ["PLANT", assigned_crop]
    harvestable, needs_water, plantable = _scan_targets(farm, board_size, plantable_crops, day)
    for group in (harvestable, needs_water, plantable):
        if not group:
            continue
        tx, ty = min(group, key=lambda t: abs(t[0] - fx) + abs(t[1] - fy))
        step = _step_toward(fx, fy, tx, ty)
        if step:
            return [step]
    return ["PASS"]

def agent(obs, configuration=None):
    farms = obs.get("farms", [])
    player = obs.get("player", 0)
    private = obs.get("private", {}) or {}
    if not farms or player >= len(farms):
        return {"farmer": ["PASS"], "hands": [], "market": []}

    farm = farms[player]
    board_size = len(farm["tiles"])
    day = obs.get("day", 0)
    money = farm["money"]
    seeds = private.get("seeds", {})
    shed = private.get("shed", {})
    market_prices = (obs.get("market", {}) or {}).get("prices", {})

    for product, price in market_prices.items():
        _update_price_history(product, price)

    portfolio = _crop_portfolio(money, market_prices)
    market_orders = []

    for product, qty in shed.items():
        if qty > 0 and product in market_prices and _should_sell(product, market_prices[product]):
            market_orders.append(["SELL", product, min(qty, SELL_BATCH_CAP)])

    for crop in portfolio:
        if seeds.get(crop, 0) < SEED_BUY_BATCH and money - CASH_RESERVE >= CROPS[crop]["seed"]:
            market_orders.append(["BUY_SEED", crop, 1])

    if len(farm["hands"]) < TARGET_CREW_SIZE and money - CASH_RESERVE >= 1:
        market_orders.append(["HIRE"])

    n_extra_unlocked = len(farm["unlocked_quadrants"]) - 1
    if n_extra_unlocked < len(LAND_ORDER):
        next_land_price = LAND_PRICES[n_extra_unlocked]
        if money - CASH_RESERVE >= next_land_price:
            market_orders.append(["BUY_LAND"])

    def crop_for_unit(idx):
        if not portfolio:
            return None
        return portfolio[idx % len(portfolio)]

    plantable_crops = bool(portfolio)
    farmer_crop = crop_for_unit(0)
    farmer_have_seed = bool(farmer_crop) and seeds.get(farmer_crop, 0) > 0
    farmer_action = _unit_action(farm["farmer"], farm, board_size, day, farmer_crop, farmer_have_seed, plantable_crops)

    hands_actions = []
    for i, hand_pos in enumerate(farm["hands"], start=1):
        hand_crop = crop_for_unit(i)
        hand_have_seed = bool(hand_crop) and seeds.get(hand_crop, 0) > 0
        hands_actions.append(_unit_action(hand_pos, farm, board_size, day, hand_crop, hand_have_seed, plantable_crops))

    return {"farmer": farmer_action, "hands": hands_actions, "market": market_orders}

Overwriting main.py


In [9]:
# Confirm the exported main.py plays a legal, working game before submitting.
env = make("kaggriculture", debug=True)
env.run(["main.py", "starter"])
print("Final banks:", [s.reward for s in env.steps[-1]])

Final banks: [18105.0, 3475.0]
